# Train DDPM on CIFAR-10 (Google Colab)

This notebook is a **thin runner** — all training logic lives in `src/`.

**Steps:**
1. Enable GPU runtime (Runtime → Change runtime type → GPU)
2. Clone / mount the repository
3. Install requirements
4. Verify CUDA
5. Import `Trainer` and start training
6. Plot loss history
7. List saved checkpoints

Do **not** duplicate the training loop here.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repository

Replace `YOUR_REPO_URL` with your GitHub/GitLab URL, or upload the `ddpm-from-scratch` folder to Drive and mount it.

In [ ]:
# Option A: clone from git
# !git clone YOUR_REPO_URL ddpm-from-scratch
# %cd ddpm-from-scratch

# Option B: Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# %cd /content/drive/MyDrive/ddpm-from-scratch

import os
from pathlib import Path

# If you already uploaded the project next to this notebook:
ROOT = Path.cwd()
if (ROOT / "src").exists():
    PROJECT = ROOT
elif (ROOT.parent / "src").exists():
    PROJECT = ROOT.parent
else:
    PROJECT = ROOT  # adjust after clone/mount

os.chdir(PROJECT)
print("Project root:", PROJECT.resolve())

## 3. Install requirements

In [ ]:
!pip install -q -r requirements.txt

## 4. Verify CUDA

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 5. Import Trainer and start training

Default: 100 epochs, batch size 128, AdamW lr=2e-4, T=1000.
For a quick smoke test, set `cfg.epochs = 1`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.config import TrainConfig
from src.training.trainer import Trainer

cfg = TrainConfig()
# cfg.epochs = 1  # uncomment for a quick smoke test
# cfg.num_workers = 2

trainer = Trainer(cfg)
loss_history = trainer.train()

## 6. Plot training loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(trainer.loss_history) + 1), trainer.loss_history)
plt.xlabel("Epoch")
plt.ylabel("Average MSE loss")
plt.title("DDPM training loss (noise prediction)")
plt.grid(True, alpha=0.3)
plt.show()

## 7. Saved checkpoint locations

In [ ]:
from pathlib import Path

ckpt_dir = Path(cfg.checkpoint_dir)
ckpts = sorted(ckpt_dir.glob("ddpm_epoch_*.pt"))
print(f"Checkpoint directory: {ckpt_dir.resolve()}")
for p in ckpts:
    print(" -", p.name, f"({p.stat().st_size / 1e6:.1f} MB)")